# Velocity & Backlog Projection

Pivotal-Tracker-style velocity tracking for GitHub Project
[#2](https://github.com/orgs/greenearth-social/projects/2/views/3).

All computation lives in the `velocity` library (unit-tested); this notebook is
presentation only.

**Prerequisite:** the `gh` CLI must be authenticated with the project scope:

```
gh auth refresh -s read:project
```

**Known limitation:** the GitHub GraphQL API returns project items in the
board's manual order, which we treat as priority order. Per-view (view #3)
custom sorting is not exposed by the API.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root, so `velocity` imports

import pandas as pd
import plotly.graph_objects as go

from velocity import github_project, velocity as vel, burndown

items = github_project.fetch_items()
print(f"Fetched {len(items)} project items")

Fetched 407 project items


## Velocity
Average completed points per week over the last 3 weeks.

In [2]:
v = vel.compute_velocity(items, weeks=3)
recent = vel.completed_weeks(items, weeks=3)
print(f"Velocity: {v:.1f} points / week")
for wk, pts in recent:
    print(f"  week of {wk}: {pts} pts")

Velocity: 20.3 points / week
  week of 2026-06-01: 20 pts
  week of 2026-06-08: 15 pts
  week of 2026-06-15: 26 pts


## Completed points per week (last ~2 months)

In [3]:
hist = burndown.completed_history(items, weeks_back=8)
fig = go.Figure(go.Bar(x=[str(wk) for wk, _ in hist], y=[pts for _, pts in hist]))
fig.add_hline(y=v, line_dash="dash", annotation_text=f"velocity {v:.1f}")
fig.update_layout(title="Completed points per week", xaxis_title="week", yaxis_title="points")
fig.show()

## Backlog burndown
Historical completion followed by a forward projection at current velocity.

In [4]:
proj = burndown.project_burndown(items, v)
total = burndown.backlog_total(items)
fig = go.Figure(go.Scatter(
    x=[str(wk) for wk, _ in proj],
    y=[rem for _, rem in proj],
    mode="lines+markers",
    name="projected remaining",
))
finish = proj[-1][0] if len(proj) > 1 else None
title = f"Backlog burndown ({total} pts" + (f", done ~{finish})" if finish else ")")
fig.update_layout(title=title, xaxis_title="week", yaxis_title="points remaining")
fig.show()

## Projected completion by week

In [5]:
backlog = burndown.backlog_items(items)
assignments = burndown.assign_tasks_to_weeks(backlog, v)
from velocity import points
df = pd.DataFrame([
    {"week": str(wk), "points": points.normalize(item), "type": points.effective_type(item), "title": item.title}
    for wk, item in assignments
])
df

,week,points,type,title
0,2026-06-22,0,bug,Metrics not reported for megastream ingest
1,2026-06-22,0,bug,Investigate (and maybe correct) low author div...
2,2026-06-22,2,feature,set an icon on our feeds
3,2026-06-22,2,feature,**** MVP demarcator ***
4,2026-06-22,0,bug,Feed is still often broken when it first loads
...,...,...,...,...
107,2026-08-31,2,feature,Script to find a GPU VM across multiple region...
108,2026-08-31,2,feature,Filtering based on megastream classifier infer...
109,2026-08-31,2,feature,Handle recency in model
110,2026-08-31,2,feature,think-aloud user testing setup & recruitment
